# 02 — Grouped model evaluation

Оценка Bi/Cr/Fe по сериям, с раздельным хранением C01/C02. Ноутбук вызывает функции train_models.py. Отдельные окна и каналы одной серии не разделяются между train/test. Независимость физических образцов и общие сеансы требуют проверки по журналу.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

cwd = Path.cwd().resolve()
BASE_DIR = next((p for p in [cwd, *cwd.parents] if (p / "src" / "dataset_config.py").exists()), None)
if BASE_DIR is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks subdirectory")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
import pandas as pd
from dataset_config import DATA_DIR, DATASET_PATH, PROCESSED_DIR, FIGURES_DIR, METRICS_DIR, FILE_METADATA
print(BASE_DIR)


In [ ]:
from train_models import (load_dataset, prepare_training_data, evaluate_models,
                          analyze_clusters, fit_final_model, analyze_comparison_films,
                          save_feature_plots)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset()
display(df.head())


In [ ]:
df_train = prepare_training_data(df)
display(df_train.groupby(["material", "experiment_id", "channel"]).size().rename("n_windows").to_frame())


## Групповая оценка

Масштабирование выполняется внутри каждого fold. MLP не использует внутреннее случайное early-stopping разбиение. Если в fold нет всех классов, выполнение останавливается, а не выдаёт вводящую в заблуждение оценку.

In [ ]:
metrics, oof_predictions = evaluate_models(df_train)
display(metrics)
display(pd.read_csv(METRICS_DIR / "evaluation_splits.csv"))
for name in ("knn", "mlp"):
    display(Image(filename=str(FIGURES_DIR / f"{name}_confusion_matrix.png")))


## Исследовательская кластеризация

Номера кластеров не являются названиями материалов. Silhouette рассчитывается на ограниченной подвыборке для больших датасетов.

In [ ]:
silhouette = analyze_clusters(df_train)
print("Silhouette:", silhouette)
display(Image(filename=str(FIGURES_DIR / "kmeans_clusters.png")))


## Финальная модель и сравниваемые серии

Финальная модель обучается заново на всех разрешённых baseline-сериях. Это не дополнительная независимая оценка качества. Вероятность класса не доказывает сохранность плёнки или отсутствие эффекта облучения.

In [ ]:
final_model = fit_final_model(df_train)
comparison_predictions, comparison_summary = analyze_comparison_films(df, final_model)
display(comparison_summary)


In [ ]:
save_feature_plots(df)
for filename in ("file_level_gamma_comparison.png", "file_level_energy_comparison.png",
                 "gamma_boxplots_by_file.png", "energy_boxplots_by_file.png"):
    display(Image(filename=str(FIGURES_DIR / filename)))


## Результаты

Полные предсказания: comparison_film_predictions.csv. Краткая сводка: comparison_file_summary.csv. irradiated_film_predictions.csv содержит только строки с подтверждённым irradiated=True; при неизвестном статусе файл будет пустым с заголовками. Новые графики не удаляют старые изображения из results/figures.